In [1]:
import truststore
truststore.extract_from_ssl()

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
########################################################
        # INDEXING (OFFLINE SETUP)
########################################################

Data Injestion + Data Cleaning

In [19]:
import bs4
from enum import Enum
from pypdf import PdfReader
from typing import Any, Optional
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader

class DataType(Enum):
    LINK = "link"
    TEXT = "text"
    PDF = "pdf"

class DataInjestion:
    def __init__(self, data: Any, type: DataType):
        self.data = data
        self.type = type
        self.text: Optional[list[Document]] = self._load_text()
        self._data_cleaning()

    def _load_text(self):

        loader_map = {
            DataType.LINK: self._link_loader,
            DataType.TEXT: self._text_loader,
            DataType.PDF: self._pdf_loader,
        }

        loader = loader_map.get(self.type)
        self.text = loader()

    def _text_loader(self):
        return [Document(self.data)]
    
    def _link_loader(self):
        loader = WebBaseLoader(
            web_path=self.data,
            bs_kwargs=dict(
                parse_only = bs4.SoupStrainer(
                    class_ = ('post-content', 'post-title', 'post-header')
                )
            )
        )

        return loader.load()
    
    def _pdf_loader(self):
        pdf = PdfReader(self.data)
        docs = []

        for i, page in enumerate(pdf):
            docs.append(
                Document(
                    page_content=page.extract_text(),
                    metadata={
                        "page_number": i
                    }
                )
            )

    
    def _data_cleaning(self):
        pass

    


Chunking

In [ ]:
from enum import Enum
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

class ChunkingType(Enum):
    Fixed_Size = "fixed_size"
    Recursive = "recursive"
    Semantic_Chunking = "semantic_chunking"
    Token_Based = "token_based"
    Hierarchical = "hierarchical"
    Agentic = "agentic"                       # Currently out of scope of project
    Late_Chunking = "late_chunking"

class Chunking:
    def __init__(self, type: ChunkingType, docs: list[Document], **kwargs):
        self.type = type
        self.kwargs = kwargs
        self.docs = docs
        self.chunks = self._chunker(docs)

    def _chunker(self):
        chunking_mapping = {
            ChunkingType.Fixed_Size: self._chunk_fixed_size,
            ChunkingType.Recursive: self._chunk_recursive,
            ChunkingType.Semantic_Chunking: self._chunk_semantic,
            ChunkingType.Token_Based: self._chunk_token_based,
            ChunkingType.Agentic: self._chunk_agentic,
            ChunkingType.Hierarchical: self._chunk_hierarchical,
        }

        return chunking_mapping.get(self.type)
    
    def _chunk_fixed_size(self):
        splitter = CharacterTextSplitter(
            chunk_size = self.kwargs.get("chunk_size", 1000),
            chunk_overlap = self.kwargs.get("chunk_overlap", 50),
            separator = self.kwargs.get("separator", '\n')
        )
        return splitter.split_documents(self.docs)

    def _chunk_recursive(self): 
        splitter = RecursiveCharacterTextSplitter(
            chunk_size = self.kwargs.get("chunk_size", 1000),
            chunk_overlap = self.kwargs.get("chunk_overlap", 50),
            separator = self.kwargs.get("separator", '\n')
        )
        return splitter.split_documents(self.docs)

    def _chunk_semantic(self): 
        splitter = SemanticChunker(
            embeddings=HuggingFaceEmbeddings(),
            breakpoint_threshold_type='percentile',
            breakpoint_threshold_amount=90
        )
        return splitter.split_documents(self.docs)

    def _chunk_token_based(self): 
        splitter = TokenTextSplitter(
            chunk_size = self.kwargs.get("chunk_size", 1000),
            chunk_overlap = self.kwargs.get("chunk_overlap", 50),
        )
        return splitter.split_documents(self.docs)

    def _chunk_hierarchical(self): 
        pass


    def _chunk_agentic(self): 
        pass


Embedding + Indexing

In [ ]:
import uuid
from enum import Enum
from langchain_groq import ChatGroq
from ragatouille import RAGPretrainedModel
from langchain_core.documents import Document
from langchain_classic.storage import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers import MultiVectorRetriever

class IndexingType(Enum):
    Navie_Chunking = "navie_chunking"
    Multiple_Representation = "multiple_representation"
    Parent_Child_Indexing = "parent_child_indexing"
    Raptor = "raptor"
    COLBert = "colbert"


class Indexing:

    def __init__(self, type: IndexingType, chunks: list[Document], **kwargs):
        self.type = type
        self.chunks = chunks
        self.kwargs = kwargs
        self.retriever = None
        self.vectorstore = Chroma(collection_name="summaries", 
                                  embedding_function=HuggingFaceEmbeddings(kwargs.get("embedding_model", "all-MiniLM-L6-v2")))
        self._indexing()

    def _indexing(self):

        indexing_map = {
            IndexingType.Navie_Chunking: self.navie_chunking(),
            IndexingType.Multiple_Representation: self.multiple_representation(),
            IndexingType.Parent_Child_Indexing: self.parent_child_indexing(),
            IndexingType.Raptor: self.raptor(),
            IndexingType.COLBert: self.colbert(),
        }

        return indexing_map.get(self.type)
    
    def navie_chunking(self):
        self.vectorstore.aadd_documents(self.chunks)
    
    def multiple_representation(self):
        id_key = "doc_id"
        model = self.kwargs.get("model", "llama-3.3-70b-versatile")
        temperature = self.kwargs.get("temperature", 0)
        chain = (
            {"doc": lambda x: x}
            | ChatPromptTemplate.from_template("Summarize following document: \n\n {doc}")
            | ChatGroq(model=model, temperature=temperature)
            | StrOutputParser()
        )

        summaries = chain.batch(self.chunks, {"max_concurrency": 5})
        docstore = InMemoryStore()
        self.retriever = MultiVectorRetriever(
            vectorstore=self.vectorstore,
            docstore=docstore,
            id_key=id_key,
        )
        doc_ids = [str(uuid.uuid4()) for _ in self.chunks]

        summary_docs = [
            Document(page_content=s, metadata={id_key: doc_ids[i]})
            for i, s in enumerate(summaries)
        ]
        self.retriever.vectorstore.add_documents(summary_docs)
        self.retriever.docstore.mset(list(zip(doc_ids, self.chunks)))

    def parent_child_indexing(self):
        id_key = "doc_id"
        child_chunk_size = self.kwargs.get("child_chunk_size", 250)
        child_chunk_overlap = self.kwargs.get("child_chunk_overlap", 50)
        doc_ids = [str(uuid.uuid4()) for _ in self.chunks]
        docstore = InMemoryStore()
        self.retriever = MultiVectorRetriever(
            vectorstore=self.vectorstore,
            docstore=docstore,
            id_key=id_key,
        )
        child_docs = [
            Document(page_content=chunk, metadata={id_key: doc_id})
            for doc, doc_id in zip(self.chunks, doc_ids)
            for chunk in Chunking(CharacterTextSplitter, [doc], chunk_size=child_chunk_size, chunk_overlap=child_chunk_overlap).chunks
        ]
        self.retriever.vectorstore.add_documents(child_docs)
        self.retriever.docstore.mset(list(zip(doc_ids, self.chunks)))

    def raptor(self):
        # check indexing .ipynb in same folder for raptor
        pass

    def colbert(self):
        RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
        RAG.index(
            collection=self.chunks,
            index_name="COLBert",
            max_document_length=180,
            split_documents=True,
        )
        return RAG




In [ ]:
########################################################
        # Quering Time (ONLINE SETUP)
########################################################

Query Transformation

In [ ]:
from enum import Enum
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class QueryEmbeddingType(Enum):
    Query_ReWritting = "query_rewritting"
    HyDE = "hyde"
    Multi_Query = "multi_query"
    Step_Back = "step_back"
    Query_Decomposition = "query_decomposition"
    RAG_Fusion = "rag_fusion"

class QueryEmbedding:

    def __init__(self, type: QueryEmbeddingType, **kwargs):
        self.type = type 
        self.kwargs = kwargs

    def _query_embedding(self, query):

        query_embedding_map = {
            QueryEmbeddingType.Query_ReWritting: self.query_rewritting(query),
            QueryEmbeddingType.HyDE: self.hyde(query),
            QueryEmbeddingType.Multi_Query: self.multi_query(query),
            QueryEmbeddingType.Step_Back: self.step_back(query),
            QueryEmbeddingType.Query_Decomposition: self.query_decomposition(query),
            QueryEmbeddingType.RAG_Fusion: self.rag_fusion(query),
        }
        return query_embedding_map.get(self.type)
    
    def query_generator(self, query, prompt):
        llm = ChatGroq(
            model=self.kwargs.get("model", "llama-3.3-70b-versatile"),
            temperature=self.kwargs.get("temperature", 0)
        )

        query_rewriter = (
            prompt
            | llm
            | StrOutputParser()
            | (lambda x: x.split("\n"))
        )

        return query_rewriter.invoke(query)
    

    def query_rewritting(self, query):
        rewrite_prompt = ChatPromptTemplate.from_template("""
            Rewrite the following query for better document retrieval.
            Make it specific and self-contained.

            Query:
            {query}
            """
        )

        return self.query_generator(query, rewrite_prompt)
    

    def hyde(self, query):
        hyde_prompt = ChatPromptTemplate.from_template("""
            Write a detailed passage answering the question.

            Question:
            {query}
            """
        )

        return self.query_generator(query, hyde_prompt)
    

    def multi_query(self, query):
        template = """You are an AI language model assistant. Your task is to generate five 
            different versions of the given user question to retrieve relevant documents from a vector 
            database. By generating multiple perspectives on the user question, your goal is to help
            the user overcome some of the limitations of the distance-based similarity search. 
            Provide these alternative questions separated by newlines. Original question: {question}"""
        prompt_perspectives = ChatPromptTemplate.from_template(template)
        return self.query_generator(query, prompt_perspectives)
    

    def step_back(self, query):
        step_back_prompt = ChatPromptTemplate.from_template("""
            Given a specific question, generate a broader,
            more general question that captures the higher-level concept.

            Question:
            {query}
            """)
        return self.query_generator(query, step_back_prompt)
    

    def query_decomposition(self, query):
        decomposition_prompt = ChatPromptTemplate.from_template("""
            Break the following question into smaller
            independent sub-questions for retrieval.

            Question:
            {query}
            """)
        return self.query_generator(query, decomposition_prompt)
    

    def rag_fusion(self, query):
        # multiple query retrieval results ko intelligently fuse karo
        # using ranking aggregation.
        pass
    

Retrival

In [ ]:
import numpy as np
from enum import Enum
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

class SearchingType(Enum):
    BM25 = "bm25"
    Semantic_Search = "semantic_search"
    Hybrid = "hybrid"
    RRF = "rrf"
    Cross_Encoder = "cross_encoder"
    Splade = "splade"

class Seaching:
    def __init__(self, type: SearchingType, **kwargs):
        self.type = type
        self.kwargs = kwargs

    def search_context(self, query: str):
        searching_map = {
            SearchingType.BM25: self.bm25(query),
            SearchingType.Semantic_Search: self.semantic_search(query),
            SearchingType.Hybrid: self.hybrid(query),
            SearchingType.RRF: self.rrf(query),
            SearchingType.Cross_Encoder: self.cross_encoder(query),
            SearchingType.Splade: self.splade(query),
        }

        return searching_map.get(self.type)
    
    def bm25(self, query: list[str], n: int = None):
        n = n or self.kwargs.get("n") or 3
        docs = self.kwargs.get("docs", [])
        tokenized_docs = [doc.lower().split() for doc in docs]
        bm25_obj = BM25Okapi(tokenized_docs)

        tokenized_query = query.lower().split()
        top_docs = bm25_obj.get_top_n(
            tokenized_query,
            docs,
            n=n
        )
        return top_docs

    def semantic_search(self, query, n: int = None):
        n = n or self.kwargs.get("n", 3)
        vectorstore = self.kwargs.get("vectorstore")
        return vectorstore.similarity_search(
            query,
            k=2
        )
    
    def hybrid(self, query):
        # formula = alpha * bm25 + (1-alpha) * self.semantic_search
        pass

    def rrf(self, query):
        # retrieval_lists = [
        #     bm25_results,
        #     dense_results
        # ]
        # for docs in results:

        #     for rank, doc in enumerate(docs):

        #         scores[doc] += 1 / (k + rank + 1)

        # reranked = sorted(
        #     scores.items(),
        #     key=lambda x: x[1],
        #     reverse=True
        # )
        pass

    def cross_encoder(self, query):
        bi_encoder    = SentenceTransformer('all-MiniLM-L6-v2')
        cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

        docs = self.kwargs.get("docs", [])
        doc_embs = bi_encoder.encode(docs, normalize_embeddings=True)
        q_emb = bi_encoder.encode([query], normalize_embeddings=True)

        bi_scores  = np.dot(doc_embs, q_emb.T).flatten()
        top20_idx  = np.argsort(bi_scores)[::-1][:20]
        candidates = [docs[i] for i in top20_idx]
        pairs = [[query, doc] for doc in candidates]

        cross_scores = cross_encoder.predict(pairs)
        reranked_idx = np.argsort(cross_scores)[::-1]
        final_top3   = [candidates[i] for i in reranked_idx[:3]]
        return final_top3

    def splade(self, query):
        pass


In [18]:
docs = [
    "Our refund policy allows returns within 30 days",
    "We ship orders within 2 business days",
    "To initiate a refund contact support with order ID",
    "We accept credit cards UPI and net banking",
]

search = Seaching(SearchingType.BM25, docs=docs)
x = search.search_context("refund policy")
x

['Our refund policy allows returns within 30 days',
 'We accept credit cards UPI and net banking',
 'To initiate a refund contact support with order ID']

In [ ]:
# TODO: Implement Splade
# TODO: Implement Chunk Hierarchical
# TODO: Implement Chunk agentic
# TODO: Context compression and Reordering

In [ ]:
# Maximal Marginal Relevance (MMR) Retrival technique

In [ ]:
# evaluation techniques